# Study Binning — bin_1x vs bin_2x per-donut Zernike comparison (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-05-11  
**Last Modified:** 2026-05-11  
**Status:** Draft  
**Keywords:** AOS, FAM, Danish, donut, binning, Zernike, systematics

## Description

The Danish per-donut wavefront estimator runs on the donut stamps
after they have been binned.  We use `bin_2x` (2×2 spatial binning) at
the summit because it is roughly an order of magnitude faster.  This
notebook compares per-donut Zernike fits between **bin_2x** (chunk1)
and **bin_1x** (chunk2) on the **same physical donuts** to look for a
binning-induced bias.

Both chunks cover the same date range (`20260315 – 20260327`), so the
underlying images, stamps, and donut centroids should be identical;
only the binning factor passed to Danish differs.  For each common
visit (rotator angle near zero) we match donuts across the two runs by
(detector, intra-focal centroid) via a KDTree, then plot:

1. Per-Zernike **scatter** of `zk_1x` vs `zk_2x`.  Aspect 1.0, axis
   range = 2-98 percentile of the pooled data.  Linear fit overlaid.
2. Per-Zernike **focal-plane map** of `zk_1x − zk_2x` vs `(thx, thy)`
   in OCS, looking for any systematic spatial trend.

**Output:** PDFs + a long-format parquet of matched donut pairs in
`output/study_binning/`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-05-11 | Aaron Roodman | Initial version. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Donut Matching](#matching)
6. [Scatter: zk_1x vs zk_2x](#scatter)
7. [Focal-plane maps: zk_1x − zk_2x](#maps)
8. [Output Tables](#output)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input donut parquet files (one per binning factor) -----
# Streaming parquet output of run_mktable.  Each carries one row
# *group* per visit (per-donut rows inside).  The matching
# `*_visits.parquet` sidecar must sit next to each donut parquet —
# we use it for the rotator-angle filter.
donut_parquet_2x = (
    'output/'
    'aos_fam_danish_v1_triplets_bin_2x_20260315_20260327.parquet')
donut_parquet_1x = (
    'output/'
    'aos_fam_danish_v1_triplets_bin_1x_20260315_20260327.parquet')

# Coordinate system for the per-donut Zernike values and field angles.
coord_sys = 'OCS'

# ----- Visit filter -----
# Restrict to visits with |rotator_angle| <= rot_max_deg so the two
# runs sample equivalent on-sky geometry (OCS and CCS coincide near
# rot = 0).
rot_max_deg = 3.0
# Optional science_program filter (None = no constraint).
program_filter = None

# ----- Donut matching -----
# Two donuts match when their (centroid_x_intra, centroid_y_intra) lie
# within `match_tol_pix` of each other on the same CCD.  Intra-focal
# centroids are used as the join key (they're identical between the
# two runs in the original full-resolution images; binning only
# affects the stamp passed to Danish).
match_tol_pix = 5.0

# ----- Plot styling -----
# Scatter: per-Zernike percentile range (2..98 %).  Same range for x
# and y so the unit-slope line is on the diagonal.
scatter_plo_pct = 2.0
scatter_phi_pct = 98.0
scatter_panels_per_page = 4
# Focal-plane map: bin size and per-Zernike color range percentile.
map_n_bins      = 24
map_fp_radius   = 1.8
map_plo_pct     = 2.0
map_phi_pct     = 98.0
map_panels_per_page = 4

# ----- Output -----
output_dir = 'output/study_binning'
output_pdf_scatter = f'{output_dir}/study_binning_scatter.pdf'
output_pdf_maps    = f'{output_dir}/study_binning_focalmaps.pdf'
output_table_path  = f'{output_dir}/study_binning_pairs.parquet'

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial import cKDTree
from scipy.stats import binned_statistic_2d
from astropy.table import QTable
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
from common.zernike_names import NOLL_NAMES

setup_plotting()

print('Ready.')

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
def _alt_to_deg(alt_arr):
    a = np.asarray(alt_arr, dtype=float)
    if np.nanmax(np.abs(a)) < 2.0 * np.pi + 1e-3:
        return np.rad2deg(a)
    return a


def visits_sidecar_path(donut_parquet_path):
    """Return the matching `*_visits.parquet` for a donut parquet."""
    p = Path(donut_parquet_path)
    return p.with_name(p.stem + '_visits.parquet')


def select_visits(visits_table, *, rot_max_deg=3.0, program_filter=None):
    """Boolean mask over visits_table for visits to keep."""
    n = len(visits_table)
    keep = np.ones(n, dtype=bool)
    if 'rotator_angle' in visits_table.colnames:
        rot = np.asarray(visits_table['rotator_angle'], dtype=float)
        keep &= np.isfinite(rot) & (np.abs(rot) <= rot_max_deg)
    if program_filter is not None and 'science_program' in visits_table.colnames:
        sp = np.asarray(visits_table['science_program']).astype(str)
        if isinstance(program_filter, (list, tuple, set)):
            prog_set = {str(p) for p in program_filter}
            keep &= np.array([s in prog_set for s in sp])
        else:
            keep &= (sp == str(program_filter))
    return keep


def build_row_group_lookup(parquet_path):
    """`(day_obs, seq_num) -> row_group_idx` from parquet column statistics.

    Mirrors the trick used in dz_fitting.fit_focal_zernikes_streaming.
    """
    pf = pq.ParquetFile(str(parquet_path))
    lookup = {}
    for i in range(pf.num_row_groups):
        meta = pf.metadata.row_group(i)
        d = s = None
        for ci in range(meta.num_columns):
            cmeta = meta.column(ci)
            name = cmeta.path_in_schema
            if name == 'day_obs' and cmeta.statistics is not None:
                d = cmeta.statistics.min
            elif name == 'seq_num' and cmeta.statistics is not None:
                s = cmeta.statistics.min
        if d is not None and s is not None:
            lookup[(int(d), int(s))] = i
    return pf, lookup


def read_visit_donuts(pf, row_group_idx):
    """Load one visit's donuts as a pandas DataFrame from row group idx."""
    return pf.read_row_group(row_group_idx).to_pandas()


def match_donuts_per_ccd(df_a, df_b, *, tol_pix=5.0):
    """Match donuts in df_a and df_b on the same CCD using intra centroids.

    Returns parallel integer index arrays (idx_a, idx_b) into the two
    input DataFrames such that each (i_a, i_b) pair refers to the same
    physical donut.
    """
    idx_a_all = []
    idx_b_all = []
    # cKDTree per detector
    det_a = df_a['detector'].to_numpy(dtype=str)
    det_b = df_b['detector'].to_numpy(dtype=str)
    for det in np.unique(det_a):
        ai = np.where(det_a == det)[0]
        bi = np.where(det_b == det)[0]
        if len(ai) == 0 or len(bi) == 0:
            continue
        xa = df_a['centroid_x_intra'].to_numpy(dtype=float)[ai]
        ya = df_a['centroid_y_intra'].to_numpy(dtype=float)[ai]
        xb = df_b['centroid_x_intra'].to_numpy(dtype=float)[bi]
        yb = df_b['centroid_y_intra'].to_numpy(dtype=float)[bi]
        tree = cKDTree(np.column_stack([xa, ya]))
        dist, k = tree.query(np.column_stack([xb, yb]), distance_upper_bound=tol_pix)
        good = np.isfinite(dist) & (dist < tol_pix)
        idx_a_all.append(ai[k[good]])
        idx_b_all.append(bi[good])
    if not idx_a_all:
        return np.empty(0, dtype=int), np.empty(0, dtype=int)
    return np.concatenate(idx_a_all), np.concatenate(idx_b_all)


def zk_label(j):
    """Pretty label 'Z{j} ({name})' for a pupil Noll index."""
    return f'Z{j} ({NOLL_NAMES.get(j, "?")})'


def stream_pdf_pages(panel_iter, panels_per_page, ncols, page_size,
                     suptitle_fmt, output_pdf, plot_panel):
    """Generic streaming PDF writer.

    `panel_iter` yields (page_payload, panel_payload) tuples; we group
    them into pages and call `plot_panel(ax, panel_payload)` per axis.
    Pages are closed after savefig so memory stays bounded.
    """
    Path(output_pdf).parent.mkdir(parents=True, exist_ok=True)
    panels = list(panel_iter)
    n_total = len(panels)
    n_pages = (n_total + panels_per_page - 1) // panels_per_page
    with PdfPages(output_pdf) as pdf:
        for page in range(n_pages):
            chunk = panels[page * panels_per_page:
                            (page + 1) * panels_per_page]
            nrows = (len(chunk) + ncols - 1) // ncols
            fig, axes = plt.subplots(
                nrows, ncols, figsize=page_size, layout='constrained')
            axes = np.atleast_1d(axes).ravel()
            for ax, panel in zip(axes, chunk):
                plot_panel(ax, panel)
            for ax in axes[len(chunk):]:
                ax.set_visible(False)
            fig.suptitle(suptitle_fmt.format(page=page + 1,
                                              n_pages=n_pages),
                         fontsize=12)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
    print(f'Wrote {n_pages} pages to {output_pdf}')
    return n_pages

<a id='data'></a>
## 4. Data Access

In [ ]:
for p in (donut_parquet_2x, donut_parquet_1x):
    if not Path(p).exists():
        raise FileNotFoundError(p)
    sidecar = visits_sidecar_path(p)
    if not sidecar.exists():
        raise FileNotFoundError(sidecar)

# Load visit sidecars and filter to the visits we want.
visits_2x_full = QTable.read(str(visits_sidecar_path(donut_parquet_2x)))
visits_1x_full = QTable.read(str(visits_sidecar_path(donut_parquet_1x)))
print(f'bin_2x visits sidecar: {len(visits_2x_full)} rows')
print(f'bin_1x visits sidecar: {len(visits_1x_full)} rows')

m2 = select_visits(visits_2x_full, rot_max_deg=rot_max_deg,
                   program_filter=program_filter)
m1 = select_visits(visits_1x_full, rot_max_deg=rot_max_deg,
                   program_filter=program_filter)
visits_2x = visits_2x_full[m2]
visits_1x = visits_1x_full[m1]
print(f'After rot/program filter: bin_2x kept {len(visits_2x)}, '
      f'bin_1x kept {len(visits_1x)}')

keys_2x = set(zip(np.asarray(visits_2x['day_obs']).astype(int).tolist(),
                   np.asarray(visits_2x['seq_num']).astype(int).tolist()))
keys_1x = set(zip(np.asarray(visits_1x['day_obs']).astype(int).tolist(),
                   np.asarray(visits_1x['seq_num']).astype(int).tolist()))
common_visits = sorted(keys_2x & keys_1x)
print(f'\nCommon visits in both runs: {len(common_visits)}')
if not common_visits:
    raise RuntimeError('No common visits found — check input files / cuts.')

# Probe Noll j list from a sample row group on the 2x file (visits
# sidecar carries the canonical list).
if 'nollIndices' in visits_2x.colnames:
    iZs = [int(j) for j in
           np.asarray(visits_2x['nollIndices'][0]).tolist()]
else:
    pf_probe = pq.ParquetFile(str(donut_parquet_2x))
    df_probe = pf_probe.read_row_group(0).to_pandas()
    nZk = len(df_probe['zk_OCS'].iloc[0])
    # Default mapping for nZk=21: Z4..Z19 + Z22..Z26.
    if nZk == 21:
        iZs = list(range(4, 20)) + list(range(22, 27))
    else:
        iZs = list(range(4, 4 + nZk))
print(f'Pupil Noll j (n={len(iZs)}): {iZs}')

<a id='matching'></a>
## 5. Donut Matching

For each common visit, stream the donut row group from each parquet,
match donuts within the same CCD by their intra-focal centroid via a
KDTree (tolerance = `match_tol_pix`), and accumulate per-donut paired
`(zk_2x, zk_1x)` arrays + the OCS field angle.

Memory-cheap: only the matched pair arrays (per-donut, per-Zernike)
are kept after each visit is processed.

In [ ]:
pf2, rg2 = build_row_group_lookup(donut_parquet_2x)
pf1, rg1 = build_row_group_lookup(donut_parquet_1x)

zk_2x_acc = []
zk_1x_acc = []
thx_acc   = []
thy_acc   = []
det_acc   = []
dobs_acc  = []
snum_acc  = []
n_matched_per_visit = []
n_unmatched_per_visit = []

zk_col       = f'zk_{coord_sys}'
thx_col_ext  = f'thx_{coord_sys}_extra'
thy_col_ext  = f'thy_{coord_sys}_extra'

print(f'Streaming {len(common_visits)} visits...')
for d, s in tqdm(common_visits, desc='matching'):
    if (d, s) not in rg2 or (d, s) not in rg1:
        continue
    df2 = read_visit_donuts(pf2, rg2[(d, s)])
    df1 = read_visit_donuts(pf1, rg1[(d, s)])
    # Require matched intra/extra in both runs.
    df2 = df2[df2['matched_intra_extra'].fillna(False)].reset_index(drop=True)
    df1 = df1[df1['matched_intra_extra'].fillna(False)].reset_index(drop=True)
    if len(df2) == 0 or len(df1) == 0:
        continue
    i2, i1 = match_donuts_per_ccd(df2, df1, tol_pix=match_tol_pix)
    n_matched = len(i2)
    n_unmatched = max(len(df2), len(df1)) - n_matched
    n_matched_per_visit.append(n_matched)
    n_unmatched_per_visit.append(n_unmatched)
    if n_matched == 0:
        continue
    zk2 = np.stack(df2[zk_col].values[i2])
    zk1 = np.stack(df1[zk_col].values[i1])
    # OCS field angles in degrees from the extra-focal centroid.
    thx = np.rad2deg(df2[thx_col_ext].to_numpy(dtype=float)[i2])
    thy = np.rad2deg(df2[thy_col_ext].to_numpy(dtype=float)[i2])
    det = df2['detector'].to_numpy(dtype=str)[i2]
    zk_2x_acc.append(zk2)
    zk_1x_acc.append(zk1)
    thx_acc.append(thx); thy_acc.append(thy)
    det_acc.append(det)
    dobs_acc.append(np.full(n_matched, d, dtype=int))
    snum_acc.append(np.full(n_matched, s, dtype=int))

zk_2x = np.concatenate(zk_2x_acc, axis=0) if zk_2x_acc else np.empty((0, len(iZs)))
zk_1x = np.concatenate(zk_1x_acc, axis=0) if zk_1x_acc else np.empty((0, len(iZs)))
thx   = np.concatenate(thx_acc) if thx_acc else np.empty(0)
thy   = np.concatenate(thy_acc) if thy_acc else np.empty(0)
det   = np.concatenate(det_acc) if det_acc else np.empty(0, dtype=str)
dobs  = np.concatenate(dobs_acc) if dobs_acc else np.empty(0, dtype=int)
snum  = np.concatenate(snum_acc) if snum_acc else np.empty(0, dtype=int)

total_matched = int(zk_2x.shape[0])
total_unmatched = int(sum(n_unmatched_per_visit))
print(f'\nMatched donut pairs: {total_matched:,}')
print(f'Unmatched donuts (per visit, in either run):  '
      f'{total_unmatched:,}  (≈ {total_unmatched / max(1, total_matched + total_unmatched):.1%})')
if n_matched_per_visit:
    print(f'Per-visit match counts: median = '
          f'{int(np.median(n_matched_per_visit))}, '
          f'range = [{min(n_matched_per_visit)}, '
          f'{max(n_matched_per_visit)}]')

<a id='scatter'></a>
## 6. Scatter: zk_1x vs zk_2x

Per Zernike `j`, one square panel showing `zk_1x` on the y-axis vs
`zk_2x` on the x-axis.  Axes are square (aspect 1.0) and clipped to
the 2–98 percentile of the pooled distribution so a few outliers don't
compress the bulk.  Black dashed line = unit slope; red solid = OLS
linear fit `y = m·x + b` annotated with the slope, intercept, and
Pearson r.

In [ ]:
def _scatter_panel(ax, payload):
    j, x, y = payload
    if len(x) == 0:
        ax.set_visible(False)
        return
    pooled = np.concatenate([x, y])
    pooled = pooled[np.isfinite(pooled)]
    if pooled.size == 0:
        ax.set_visible(False)
        return
    lo = float(np.nanpercentile(pooled, scatter_plo_pct))
    hi = float(np.nanpercentile(pooled, scatter_phi_pct))
    pad = 0.02 * max(abs(hi - lo), 1e-6)
    lo -= pad; hi += pad
    # Scatter
    ax.plot(x, y, '.', markersize=2, alpha=0.25, color='tab:blue',
            rasterized=True)
    # Unit slope reference
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.7)
    # OLS fit (clipped to range used for plotting so wild outliers don't
    # distort the line)
    mask = (np.isfinite(x) & np.isfinite(y)
            & (x >= lo) & (x <= hi) & (y >= lo) & (y <= hi))
    if int(mask.sum()) >= 5:
        coef = np.polyfit(x[mask], y[mask], 1)
        m, b = float(coef[0]), float(coef[1])
        xf = np.array([lo, hi])
        ax.plot(xf, m * xf + b, 'r-', lw=1.2, alpha=0.85)
        r = float(np.corrcoef(x[mask], y[mask])[0, 1])
        label = (f'slope={m:.3f}\nintercept={b:+.4f} μm\nr={r:+.4f}'
                  f'\nn={int(mask.sum())}')
    else:
        label = f'n={int(np.isfinite(x).sum())}'
    ax.text(0.04, 0.96, label, transform=ax.transAxes, ha='left',
            va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', alpha=0.85,
                      lw=0))
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('zk_2x [μm]', fontsize=9)
    ax.set_ylabel('zk_1x [μm]', fontsize=9)
    ax.set_title(zk_label(j), fontsize=10)
    ax.tick_params(labelsize=8)


scatter_panels = [
    (j, zk_2x[:, idx], zk_1x[:, idx])
    for idx, j in enumerate(iZs)
]
n_scatter_pages = stream_pdf_pages(
    scatter_panels,
    panels_per_page=scatter_panels_per_page, ncols=2,
    page_size=(10, 10),
    suptitle_fmt=(f'bin_1x vs bin_2x per-donut Zernike scatter  '
                  f'(n_pairs={total_matched:,}, |rot|<{rot_max_deg:g}°)  '
                  f'  [{{page}}/{{n_pages}}]'),
    output_pdf=output_pdf_scatter,
    plot_panel=_scatter_panel)

<a id='maps'></a>
## 7. Focal-plane maps: `zk_1x − zk_2x`

Per Zernike `j`, a coarse `(thx, thy)` binned median of `zk_1x − zk_2x`
in OCS.  Diverging colormap centered on zero; per-panel color range
from the 2–98 % percentile of the per-bin medians so the spatial
pattern (if any) stands out.  4 panels per page.

In [ ]:
_xb = np.linspace(-map_fp_radius, map_fp_radius, map_n_bins + 1)
_yb = np.linspace(-map_fp_radius, map_fp_radius, map_n_bins + 1)


def _map_panel(ax, payload):
    j, delta = payload
    if len(delta) == 0:
        ax.set_visible(False)
        return
    stat, _, _, _ = binned_statistic_2d(
        thy, thx, delta, statistic='median',
        bins=[_xb, _yb])
    finite = stat[np.isfinite(stat)]
    if finite.size == 0:
        ax.set_visible(False)
        return
    lo = float(np.nanpercentile(finite, map_plo_pct))
    hi = float(np.nanpercentile(finite, map_phi_pct))
    vlim = max(abs(lo), abs(hi), 1e-6)
    im = ax.imshow(
        stat.T, origin='lower',
        extent=[_xb[0], _xb[-1], _yb[0], _yb[-1]],
        cmap='RdBu_r', interpolation='none', aspect='equal',
        vmin=-vlim, vmax=vlim)
    plt.colorbar(im, ax=ax, shrink=0.75, label='Δ [μm]')
    med = float(np.nanmedian(delta))
    mad = float(np.nanmedian(np.abs(delta - med)))
    sigma_mad = 1.4826 * mad
    ax.text(0.04, 0.96,
            f'all-FOV median = {med:+.4f}\n'
            f'σ_MAD = {sigma_mad:.4f}\n'
            f'n_donuts = {int(np.isfinite(delta).sum())}',
            transform=ax.transAxes, ha='left', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', alpha=0.85,
                      lw=0))
    ax.set_xlabel(f'thy_{coord_sys} [deg]', fontsize=9)
    ax.set_ylabel(f'thx_{coord_sys} [deg]', fontsize=9)
    ax.set_title(zk_label(j) + '  —  bin_1x − bin_2x', fontsize=10)
    ax.tick_params(labelsize=8)


map_panels = [
    (j, zk_1x[:, idx] - zk_2x[:, idx])
    for idx, j in enumerate(iZs)
]
n_map_pages = stream_pdf_pages(
    map_panels,
    panels_per_page=map_panels_per_page, ncols=2,
    page_size=(11, 10),
    suptitle_fmt=(f'Focal-plane Δzk = bin_1x − bin_2x  '
                  f'(n_pairs={total_matched:,}, '
                  f'{map_n_bins}×{map_n_bins} bins, '
                  f'|rot|<{rot_max_deg:g}°)  [{{page}}/{{n_pages}}]'),
    output_pdf=output_pdf_maps,
    plot_panel=_map_panel)

<a id='output'></a>
## 8. Output Tables

Long-format parquet with one row per matched donut pair, carrying
(day_obs, seq_num, detector, thx_OCS, thy_OCS) plus every per-Zernike
`zk_2x_z{j}`, `zk_1x_z{j}`, and `delta_z{j} = zk_1x − zk_2x` column.

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
out = {
    'day_obs':     dobs,
    'seq_num':     snum,
    'detector':    det,
    'thx_OCS_deg': thx,
    'thy_OCS_deg': thy,
}
for idx, j in enumerate(iZs):
    out[f'zk_2x_z{j}']  = zk_2x[:, idx]
    out[f'zk_1x_z{j}']  = zk_1x[:, idx]
    out[f'delta_z{j}']  = zk_1x[:, idx] - zk_2x[:, idx]
df_out = pd.DataFrame(out)
df_out.to_parquet(output_table_path)
print(f'Saved {len(df_out):,} matched donut pairs -> {output_table_path}')
print(f'Columns: {len(df_out.columns)}')

# Quick per-Zernike summary table (median, σ_MAD, mean) of the
# bin_1x - bin_2x deltas.
summary_rows = []
for idx, j in enumerate(iZs):
    delta = zk_1x[:, idx] - zk_2x[:, idx]
    delta = delta[np.isfinite(delta)]
    if len(delta) == 0:
        continue
    med = float(np.median(delta))
    mad = float(np.median(np.abs(delta - med)))
    summary_rows.append({
        'j': int(j),
        'name': NOLL_NAMES.get(j, '?'),
        'n': int(len(delta)),
        'mean':       float(np.mean(delta)),
        'median':     med,
        'sigma_mad':  1.4826 * mad,
    })
df_summary = pd.DataFrame(summary_rows)
with pd.option_context('display.max_rows', None,
                       'display.float_format', '{:+.4f}'.format):
    display(df_summary)